In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_legal_process_stage_qca(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "LegalProcessStageId string,"
        #     "SourceSystem string,"
        #     "PortfolioKey string,"
        #     "PortfolioName string,"
        #     "IngestionTimestamp timestamp,"
        #     "BatchId string,"
        #     "LegalProcessStageKey string,"
        #     "LegalProcessKey string,"
        #     "ExposureKey string,"
        #     "PartyKey string,"
        #     "CDI string,"
        #     "CaseCode string,"
        #     "LegalProcessCreatedBy string,"
        #     "LegalProcessCreatedDate timestamp,"
        #     "LegalProcessStartDate timestamp,"
        #     "LegalProcessEndDate string,"
        #     "LegalProcessDefinitionCode string,"
        #     "LegalProcessDefinition string,"
        #     "LegalProcessStatusCode int,"
        #     "LegalProcessStatus string,"
        #     "ExternalCompanyAssignmentKey int,"
        #     "LegalProcessReferenceNumber string,"
        #     "LegalProcessControlAreaKey int,"
        #     "LegalStageKey string,"
        #     "LegalStageCreatedBy string,"
        #     "LegalStageCreatedDate timestamp,"
        #     "LegalStageStartDate timestamp,"
        #     "LegalStageEndDate timestamp,"
        #     "LegalStageDefinitionCode int,"
        #     "LegalStageDefinition string,"
        #     "LegalStageCurrentFlag boolean,"
        #     "LegalStageFinalFlag string,"
        #     "ParalellLegalStageKey string,"
        #     "ParalellLegalStageCreatedBy string,"
        #     "ParalellLegalStageCreatedDate timestamp,"
        #     "ParalellLegalStageStartDate timestamp,"
        #     "ParalellLegalStageDefinitionCode int,"
        #     "ParalellLegalStageDefinition string,"
        #     "RelatedLegalStageKey int,"
        #     "LegalStagePiteKey int,"
        #     "LegalStagePreviousPiteKey int,"
        #     "LegalStageFspishDuration int,"
        #     "BusinessDate Date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.LegalProcessStage"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "qca",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== fs_process_instance_stages_history ==================
        fs_process_instance_stages_history_extra_filters = None

        if fs_process_instance_stages_history_extra_filters is not None:
            fs_process_instance_stages_history_extra_filters.append(date_filter)
            fs_process_instance_stages_history_final_filters = " AND ".join(
                fs_process_instance_stages_history_extra_filters
            )
        else:
            fs_process_instance_stages_history_final_filters = date_filter

        fs_process_instance_stages_history_selected_fields = [
            "pite_id_prev",
            "fspish_duration",
            "pdst_af_id",
            "pd_desc",
            "pris_is_current_flag",
            "pite_id",
            "pite_id_prev",
            "prin_id",
        ]

        fs_process_instance_stages_history = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fs_process_instance_stages_history",  # check name
            fs_process_instance_stages_history_final_filters,
            fs_process_instance_stages_history_selected_fields,
        )

        fs_process_instance_stages_history = fs_process_instance_stages_history.select(
            "pite_id_prev",
            "fspish_duration",
            "pdst_af_id",
            "pd_desc",
            "pris_is_current_flag",
            "pite_id",
            "pite_id_prev",
            "prin_id",
        ).distinct()

        # ================== fs_process_instance ==================

        fs_process_instance_extra_filters = None

        if fs_process_instance_extra_filters is not None:
            fs_process_instance_extra_filters.append(date_filter)
            fs_process_instance_final_filters = " AND ".join(
                fs_process_instance_extra_filters
            )
        else:
            fs_process_instance_final_filters = date_filter

        fs_process_instance_selected_fields = [
            "prin_id",
            "cust_id",
            "prin_created_by",
            "prin_created",
            "wkfv_end_date_id",
            "prdf_id",
            "proc_status_id",
            "extc_id",
            "prin_reference_number",
            "process_spet_id",
            "businessdate",
        ]

        fs_process_instance = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fs_process_instance",  # check name
            fs_process_instance_final_filters,
            fs_process_instance_selected_fields,
        )
        fs_process_instance = fs_process_instance.withColumnRenamed(
            "prdf_id", "prdf_id_fpi"
        )
        fs_process_instance = fs_process_instance.withColumnRenamed(
            "proc_status_id", "proc_status_id_fpi"
        )
        fs_process_instance = fs_process_instance.withColumnRenamed(
            "prin_id", "prin_id_fpi"
        )
        # ================== dim_process_status ==================

        dim_process_status_extra_filters = None

        if dim_process_status_extra_filters is not None:
            dim_process_status_extra_filters.append(date_filter)
            dim_process_status_final_filters = " AND ".join(
                dim_process_status_extra_filters
            )
        else:
            dim_process_status_final_filters = date_filter

        dim_process_status_selected_fields = ["proc_status_desc", "proc_status_id"]

        dim_process_status = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_process_status",  # check name
            dim_process_status_final_filters,
            dim_process_status_selected_fields,
        )
        # ================== dim_process_definition ==================

        dim_process_definition_extra_filters = None

        if dim_process_definition_extra_filters is not None:
            dim_process_definition_extra_filters.append(date_filter)
            dim_process_definition_final_filters = " AND ".join(
                dim_process_definition_extra_filters
            )
        else:
            dim_process_definition_final_filters = date_filter

        dim_process_definition_selected_fields = ["prdf_name", "prdf_id"]

        dim_process_definition = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_process_definition",  # check name
            dim_process_definition_final_filters,
            dim_process_definition_selected_fields,
        )
        # ================== fsad_process_instance_participants ==================

        fsad_process_instance_participants_extra_filters = None

        if fsad_process_instance_participants_extra_filters is not None:
            fsad_process_instance_participants_extra_filters.append(date_filter)
            fsad_process_instance_participants_final_filters = " AND ".join(
                fsad_process_instance_participants_extra_filters
            )
        else:
            fsad_process_instance_participants_final_filters = date_filter

        fsad_process_instance_participants_selected_fields = [
            "cust_code",
            "CASE_CODE",
            "prin_code",
        ]

        fsad_process_instance_participants = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fsad_process_instance_participants",  # check name
            fsad_process_instance_participants_final_filters,
            fsad_process_instance_participants_selected_fields,
        )
        # ================== dim_cases_static ==================

        dim_cases_static_extra_filters = None

        if dim_cases_static_extra_filters is not None:
            dim_cases_static_extra_filters.append(date_filter)
            dim_cases_static_final_filters = " AND ".join(
                dim_cases_static_extra_filters
            )
        else:
            dim_cases_static_final_filters = date_filter

        dim_cases_static_selected_fields = [
            "debt_code",
            "CASE_CODE",
            "CASE_CODE_PRIMARY",
            "CASE_ACCOUNT_FOR_SEARCH",
        ]

        dim_cases_static = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_cases_static",  # check name
            dim_cases_static_final_filters,
            dim_cases_static_selected_fields,
        )
        dim_cases_static = dim_cases_static.withColumnRenamed(
            "CASE_CODE", "CASE_CODE_dcs"
        )
        # ================== dim_process_stage_definition ==================

        dim_process_stage_definition_extra_filters = None

        if dim_process_stage_definition_extra_filters is not None:
            dim_process_stage_definition_extra_filters.append(date_filter)
            dim_process_stage_definition_final_filters = " AND ".join(
                dim_process_stage_definition_extra_filters
            )
        else:
            dim_process_stage_definition_final_filters = date_filter

        dim_process_stage_definition_selected_fields = [
            "pdst_name",
            "pdst_is_final_flag",
            "pdst_id",
        ]

        dim_process_stage_definition = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_process_stage_definition",  # check name
            dim_process_stage_definition_final_filters,
            dim_process_stage_definition_selected_fields,
        )

        # ================== portfolio ==================

        portfolio_extra_filters = [
            "PortfolioName in ('Core Retail','Core Wholesale')"
        ]

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioName",
            "portfolio_id_qca",        
        ]
        
        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )


        # ================== FS_CASES_SNAP ==================

        FS_CASES_SNAP_extra_filters = None

        if FS_CASES_SNAP_extra_filters is not None:
            FS_CASES_SNAP_extra_filters.append(date_filter)
            FS_CASES_SNAP_final_filters = " AND ".join(FS_CASES_SNAP_extra_filters)
        else:
            FS_CASES_SNAP_final_filters = date_filter

        FS_CASES_SNAP_selected_fields = [
            "CASE_CODE",
            "SPET_CODE",
        ]

        FS_CASES_SNAP = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_FS_CASES_SNAP",
            FS_CASES_SNAP_final_filters,
            FS_CASES_SNAP_selected_fields,
        )

        # ================== DIM_CUSTOMER_STATIC ==================
        DIM_CUSTOMER_STATIC_extra_filters = None

        if DIM_CUSTOMER_STATIC_extra_filters is not None:
            DIM_CUSTOMER_STATIC_extra_filters.append(date_filter)
            DIM_CUSTOMER_STATIC_final_filters = " AND ".join(
                DIM_CUSTOMER_STATIC_extra_filters
            )
        else:
            DIM_CUSTOMER_STATIC_final_filters = date_filter

        DIM_CUSTOMER_STATIC_selected_fields = [
            "cust_id",
            "CUST_CIF",
        ]

        DIM_CUSTOMER_STATIC = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_DIM_CUSTOMER_STATIC",
            DIM_CUSTOMER_STATIC_final_filters,
            DIM_CUSTOMER_STATIC_selected_fields,
        )

        # ================== Join Conditions==================
        FS_CASES_SNAP_dim_cases_static_join_condition = (
            FS_CASES_SNAP["CASE_CODE"] == dim_cases_static["CASE_CODE_dcs"]
        ) & (dim_cases_static["CASE_CODE_dcs"] == dim_cases_static["CASE_CODE_PRIMARY"])

        FS_CASES_SNAP_portfolio_join_condition = (
            FS_CASES_SNAP["SPET_CODE"] == portfolio["portfolio_id_qca"]
        )

        legal_process_stage_qca_join_condition = (
            fs_process_instance.join(
                dim_process_definition,
                (
                    fs_process_instance["prdf_id_fpi"]
                    == dim_process_definition["prdf_id"]
                ),
                "left",
            )
            .join(
                dim_process_status,
                (
                    fs_process_instance["proc_status_id_fpi"]
                    == dim_process_status["proc_status_id"]
                ),
                "left",
            )
            .join(
                fs_process_instance_stages_history,
                (
                    fs_process_instance["prin_id_fpi"]
                    == fs_process_instance_stages_history["prin_id"]
                ),
                "left",
            )
            .join(
                fsad_process_instance_participants,
                (
                    fs_process_instance["prin_id_fpi"]
                    == fsad_process_instance_participants["prin_code"]
                ),
                "left",
            )
            .join(
                dim_process_stage_definition,
                (
                    fs_process_instance_stages_history["pdst_af_id"]
                    == dim_process_stage_definition["pdst_id"]
                ),
                "left",
            )
            .join(
                dim_cases_static,
                (
                    fsad_process_instance_participants["CASE_CODE"]
                    == dim_cases_static["CASE_CODE_dcs"]
                ),
                "inner",
            )
            .join(
                FS_CASES_SNAP,
                FS_CASES_SNAP_dim_cases_static_join_condition,
                "inner",
            )
            .join(
                portfolio,
                FS_CASES_SNAP_portfolio_join_condition,
                "inner",
            )
            .join(
                DIM_CUSTOMER_STATIC,
                (
                    DIM_CUSTOMER_STATIC["cust_id"]
                    == coalesce(
                        fsad_process_instance_participants["cust_code"],
                        fs_process_instance["cust_id"],
                    )
                ),
                "left",
            )
        )

        legal_process_stage_qca_primary_key_fields = [
            coalesce(fs_process_instance.prin_id_fpi, lit("N/A")),
            coalesce(fsad_process_instance_participants.CASE_CODE, lit("N/A")),
            coalesce(fsad_process_instance_participants.cust_code, lit("N/A")),
            coalesce(fs_process_instance.cust_id, lit("N/A")),
            coalesce(fs_process_instance_stages_history.pite_id, lit("N/A")),
            coalesce(fs_process_instance_stages_history.pite_id_prev, lit("N/A")),
            coalesce(fs_process_instance_stages_history.fspish_duration, lit("N/A")),
            lit("qca"),
        ]
        LegalProcessStageKey_field = [
            coalesce(fs_process_instance.prin_id_fpi, lit("N/A")),
            coalesce(fsad_process_instance_participants.CASE_CODE, lit("N/A")),
            coalesce(fsad_process_instance_participants.cust_code, lit("N/A")),
            coalesce(fs_process_instance.cust_id, lit("N/A")),
            coalesce(fs_process_instance_stages_history.pite_id, lit("N/A")),
            coalesce(fs_process_instance_stages_history.pite_id_prev, lit("N/A")),
            coalesce(fs_process_instance_stages_history.fspish_duration, lit("N/A")),
        ]

        legal_process_stage_qca = (
            legal_process_stage_qca_join_condition.withColumn(
                "LegalProcessStageId",
                sha2(
                    concat_ws(";", *legal_process_stage_qca_primary_key_fields),
                    256,
                ),
            )
            .withColumn(
                "LegalProcessStageKey", concat_ws(";", *LegalProcessStageKey_field)
            )
            .withColumn(
                "PartyKey",
                coalesce(
                    fsad_process_instance_participants["cust_code"],
                    fs_process_instance["prin_id_fpi"],
                ),
            )
        )

        # ================== Selection of Final Columns ==================

        legal_process_stage_qca = legal_process_stage_qca.select(
            legal_process_stage_qca.LegalProcessStageId.cast("String").alias(
                "LegalProcessStageId"
            ),
            lit("qca").alias("SourceSystem"),
            portfolio.portfolio_id_qca.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            legal_process_stage_qca.LegalProcessStageKey.cast("String").alias(
                "LegalProcessStageKey"
            ),
            legal_process_stage_qca.prin_id_fpi.cast("string").alias("LegalProcessKey"),
            legal_process_stage_qca.CASE_ACCOUNT_FOR_SEARCH.cast("string").alias(
                "ExposureKey"
            ),
            legal_process_stage_qca.PartyKey.cast("string").alias("PartyKey"),
            legal_process_stage_qca.CUST_CIF.cast("string").alias("CDI"),
            legal_process_stage_qca.CASE_CODE_dcs.cast("string").alias("CaseCode"),
            legal_process_stage_qca.prin_created_by.alias("LegalProcessCreatedBy"),
            legal_process_stage_qca.prin_created_by.cast("timestamp").alias(
                "LegalProcessCreatedDate"
            ),
            legal_process_stage_qca.prin_created.alias("LegalProcessStartDate"),
            legal_process_stage_qca.wkfv_end_date_id.cast("string").alias(
                "LegalProcessEndDate"
            ),
            legal_process_stage_qca.prdf_name.cast("string").alias(
                "LegalProcessDefinitionCode"
            ),
            legal_process_stage_qca.prdf_id_fpi.cast("string").alias(
                "LegalProcessDefinition"
            ),
            legal_process_stage_qca.proc_status_id.alias("LegalProcessStatusCode"),
            legal_process_stage_qca.proc_status_desc.alias("LegalProcessStatus"),
            legal_process_stage_qca.extc_id.alias("ExternalCompanyAssignmentKey"),
            legal_process_stage_qca.prin_reference_number.alias(
                "LegalProcessReferenceNumber"
            ),
            legal_process_stage_qca.process_spet_id.alias("LegalProcessControlAreaKey"),
            legal_process_stage_qca.pdst_af_id.cast("string").alias("LegalStageKey"),
            lit(None).alias("LegalStageCreatedBy"),
            lit(None).cast("timestamp").alias("LegalStageCreatedDate"),
            legal_process_stage_qca.pd_desc.alias("LegalStageStartDate"),
            lit(None).alias("LegalStageEndDate"),
            legal_process_stage_qca.pdst_af_id.alias("LegalStageDefinitionCode"),
            legal_process_stage_qca.pdst_name.alias("LegalStageDefinition"),
            legal_process_stage_qca.pdst_is_final_flag.alias("LegalStageCurrentFlag"),
            legal_process_stage_qca.pris_is_current_flag.cast("string").alias(
                "LegalStageFinalFlag"
            ),
            lit(None).cast("string").alias("ParalellLegalStageKey"),
            lit(None).alias("ParalellLegalStageCreatedBy"),
            lit(None).alias("ParalellLegalStageCreatedDate"),
            lit(None).alias("ParalellLegalStageStartDate"),
            lit(None).alias("ParalellLegalStageDefinitionCode"),
            lit(None).alias("ParalellLegalStageDefinition"),
            lit(None).alias("RelatedLegalStageKey"),
            legal_process_stage_qca.pite_id.alias("LegalStagePiteKey"),
            legal_process_stage_qca.pite_id_prev.alias("LegalStagePreviousPiteKey"),
            legal_process_stage_qca.fspish_duration.alias("LegalStageFspishDuration"),
            fs_process_instance.businessdate.alias("BusinessDate"),
        )

        # ================== Append ==================

        legal_process_stage_qca.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")

In [0]:
load_curated_legal_process_stage_qca(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)
